In [ ]:

### training script for yolov11 run, rcnn, VIT push to repo 

### evals script for each model checkpoint against each other 

### Detailed to readme with regards to setup and how to reproduce similar results 

### results and checkpoints  

### inference script for each model 


In [ ]:
%pip install ultralytics 

In [ ]:
yaml = """ 
path: /kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset
test: test/images
train: train/images
val: val/images

names:
  0: camouflage_soldier
  1: weapon
  2: military_tank
  3: military_truck
  4: military_vehicle
  5: civilian
  6: soldier
  7: civilian_vehicle
  8: military_artillery
  9: trench
  10: military_aircraft
  11: military_warship


"""

with open("yolov11.yaml", "w") as f:
    f.write(yaml) 



In [ ]:
from ultralytics import YOLO
import wandb
import os
# os.environ["WANDB_API_KEY"] = "2c5208bfd1824ba17f280f2efcfecfa6a2c1669c"
# os.environ["WANDB_MODE"] = "offline"




# wandb.init(
#     project="artillery_detection",
#     config={
#         "model": "yolo11n",
#         "task": "artillery_detection",
#         "imgsz": 640,
#         "epochs_max": 100,
#         "dataset": "military_assets_12c",
#         "framework": "ultralytics",
#     }
# )

model = YOLO("yolo11n.pt")

results = model.train(
    data="/kaggle/input/military-assets-dataset-12-classes-yolo8-format/military_object_dataset/military_dataset.yaml",
    epochs=100,
    imgsz=640,
    device=[0, 1],
    project="runs/artillery",
    name="yolo11n",
    save=True,
    save_period=5,
    exist_ok=True
)

# wandb.finish()


## Inference Script
Run inference on the Military Assets Dataset (12 classes) test set.

In [2]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
import numpy as np
import os

CHECKPOINT_PATH = r"D:\art_detection\artillery-detection\yolov11_run2\yolov11_art_best.pt"

model = YOLO(CHECKPOINT_PATH)
print(f"Model loaded from: {CHECKPOINT_PATH}")
print(f"Model classes: {model.names}")

Model loaded from: D:\art_detection\artillery-detection\yolov11_run2\yolov11_art_best.pt
Model classes: {0: 'camouflage_soldier', 1: 'weapon', 2: 'military_tank', 3: 'military_truck', 4: 'military_vehicle', 5: 'civilian', 6: 'soldier', 7: 'civilian_vehicle', 8: 'military_artillery', 9: 'trench', 10: 'military_aircraft', 11: 'military_warship'}


In [3]:
%pip install kagglehub -q

import kagglehub
from pathlib import Path

DATASET_DIR = Path(kagglehub.dataset_download('rawsi18/military-assets-dataset-12-classes-yolo8-format'))
print(f"Dataset downloaded to: {DATASET_DIR}")

test_images = list(DATASET_DIR.glob("**/test/images/*.jpg"))
print(f"Found {len(test_images)} test images")


[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


d:\art_detection\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 54%|█████▎    | 2.05G/3.83G [32:53<28:28, 1.12MB/s]  


ConnectionError: HTTPSConnectionPool(host='storage.googleapis.com', port=443): Read timed out.

In [ ]:
results = model.predict(
    source=test_images,
    conf=0.25,
    iou=0.45,
    imgsz=640,
    save=True,
    save_txt=True,
    project="runs/inference",
    name="military_test",
    exist_ok=True
)

print(f"Inference complete! Results saved")

In [ ]:
def display_results(results, max_images=4):
    n_images = min(len(results), max_images)
    cols = 2
    rows = (n_images + 1) // 2
    
    fig, axes = plt.subplots(rows, cols, figsize=(14, 7 * rows))
    axes = axes.flatten() if n_images > 1 else [axes]
    
    for idx, result in enumerate(results[:max_images]):
        annotated = result.plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        axes[idx].axis('off')
        
        n_detections = len(result.boxes)
        classes_detected = [model.names[int(c)] for c in result.boxes.cls] if n_detections > 0 else []
        axes[idx].set_title(f"Detections: {n_detections}\n{', '.join(set(classes_detected))[:50]}", fontsize=10)
    
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

display_results(results, max_images=4)

In [ ]:
def analyze_detections(results, model):
    total_detections = 0
    class_counts = {}
    confidence_scores = []
    
    for result in results:
        boxes = result.boxes
        total_detections += len(boxes)
        
        for box in boxes:
            cls_id = int(box.cls)
            cls_name = model.names[cls_id]
            conf = float(box.conf)
            class_counts[cls_name] = class_counts.get(cls_name, 0) + 1
            confidence_scores.append(conf)
    
    print("=" * 50)
    print("INFERENCE SUMMARY")
    print("=" * 50)
    print(f"Total images processed: {len(results)}")
    print(f"Total detections: {total_detections}")
    print(f"Average detections per image: {total_detections / len(results):.2f}")
    
    if confidence_scores:
        print(f"\nConfidence Statistics:")
        print(f"  Mean: {np.mean(confidence_scores):.3f}")
        print(f"  Min:  {np.min(confidence_scores):.3f}")
        print(f"  Max:  {np.max(confidence_scores):.3f}")
    
    if class_counts:
        print(f"\nDetections by class:")
        for cls_name, count in sorted(class_counts.items(), key=lambda x: -x[1]):
            print(f"  {cls_name}: {count}")
    
    return class_counts, confidence_scores

class_counts, confidence_scores = analyze_detections(results, model)

In [ ]:
def infer_single_image(model, image_path, conf=0.25, show=True):
    result = model.predict(source=image_path, conf=conf, iou=0.45, imgsz=640, verbose=False)[0]
    
    if show:
        annotated = result.plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_rgb)
        plt.axis('off')
        plt.title(f"Detections: {len(result.boxes)}")
        plt.show()
    
    detections = []
    for box in result.boxes:
        detections.append({
            'class': model.names[int(box.cls)],
            'confidence': float(box.conf),
            'bbox_xyxy': box.xyxy[0].tolist(),
            'bbox_xywh': box.xywh[0].tolist()
        })
    return detections

if test_images:
    detections = infer_single_image(model, test_images[0])
    print("\nDetection details:")
    for det in detections:
        print(f"  {det['class']}: {det['confidence']:.2f} @ {[int(x) for x in det['bbox_xyxy']]}")

In [ ]:
DATASET_YAML = DATASET_DIR / "military_object_dataset" / "military_dataset.yaml"

metrics = model.val(
    data=str(DATASET_YAML),
    imgsz=640,
    batch=8,
    conf=0.001,
    iou=0.6,
    split="test",
    project="runs/val",
    name="military_eval",
    exist_ok=True
)

print("\n" + "=" * 50)
print("VALIDATION METRICS (Military Assets Test Set)")
print("=" * 50)
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"Precision:  {metrics.box.mp:.4f}")
print(f"Recall:     {metrics.box.mr:.4f}")

In [ ]:
#tensor rt could be used as well to optimize it for GPUS  

# model.export(format="onnx", imgsz=640, simplify=True)
# model.export(format="engine", imgsz=640, half=True)
